In [18]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.metrics import root_mean_squared_error,accuracy_score
import xgboost as xgb

In [19]:
TRAIN_PATH = './dataset/train.csv'
TEST_PATH = './dataset/test.csv'

TARGET = 'quality'
ID_COL = 'id'
assert os.path.exists(TRAIN_PATH) , "doesnt exist"
assert os.path.exists(TEST_PATH) , "doesnt exist"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

In [20]:
FEATURE_COLS = train.columns
DROP_COLS = [TARGET,ID_COL] # Afeter FEATURE ENGEENING , we update this array
USED_COLS = [c for c in FEATURE_COLS if c not in DROP_COLS]

# Pipeline

In [31]:

X = train[USED_COLS]
y = train[TARGET]

le = LabelEncoder()

y = le.fit_transform(y)
num_transformer = Pipeline(steps=[
    ('scale',StandardScaler())
])

preprocessor = ColumnTransformer(
        transformers = [
        ('num',num_transformer,USED_COLS)
    ])

model = xgb.XGBClassifier(
colsample_bytree= 0.8,
learning_rate= 0.01, 
max_depth= 5,
n_estimators= 300, 
subsample=0.8
)

pipeline = Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',model)
])

In [32]:
np.unique(y)

array([0, 1, 2, 3, 4, 5])

In [33]:
X_train , X_test , y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [34]:
pipeline.fit(X_train,y_train)

y_predict = pipeline.predict(X_test)

acc = accuracy_score(y_test,y_predict)

print(acc)

0.536


In [15]:
from sklearn.model_selection import GridSearchCV, KFold

# Define the model
xgb = xgb.XGBClassifier( random_state=42)

# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200,300,],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Define cross-validation strategy
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Set up GridSearchCV
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='accuracy',  # or 'r2', 'neg_mean_absolute_error'
    cv=cv,
    verbose=1,
    n_jobs=-1
)

# Fit the model
grid_search.fit(X_train, y_train)

# Results
print("Best Parameters:", grid_search.best_params_)
print("Best Score:", -grid_search.best_score_)  # negate to get positive MSE

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 300, 'subsample': 0.8}
Best Score: -0.5500833333333333


In [35]:
y_pred = pipeline.predict(test[USED_COLS])
y_prediction = le.inverse_transform(y_pred)

In [37]:
print(np.unique(y_prediction))

[5. 6. 7.]


In [41]:
y_pred = pipeline.predict(test[USED_COLS])

submission = pd.DataFrame({ID_COL:test[ID_COL].values,"quality":y_prediction})

SUBMISSION_PATH = 'submission.csv'

submission.to_csv(SUBMISSION_PATH,index=False)

submission.head()

,id,quality
0,15000,5.0
1,15001,6.0
2,15002,5.0
3,15003,7.0
4,15004,6.0
